# Test evaluation (v2) - one-time, with spread estimate

The only notebook that reads `splits/test_LOCKED.csv`.

**Design.** Not a single model but the **five fold models** of each architecture
score the test set. From the same CV runs this yields a genuine spread estimate
on the test set (five test predictions per architecture), without a single
additional training run. Because the epoch count from the HPO is fixed and no
checkpoint is selected on the test set, this is pure inference - no selection
happens on the test set.

**Prerequisites**

1. `build_split_v2` has run, `splits/` is complete
2. all three CV notebooks are done; `cv/<variant>/<variant>_fold{1..5}.pth`
   exist
3. the error analyses are finished - after this notebook no more model or
   analysis decisions should be made

**Three levels of statistics**

- **descriptive**: macro-F1 of the five fold models per architecture, mean +/- std
- **paired over seeds**: paired t-test and ASO test (deep-significance) over the
  five fold differences per pair
- **paired over items**: bootstrap confidence interval of the macro-F1 difference
  on the averaged (soft-vote) predictions, plus exact McNemar on the majority
  votes

We always report the **confidence interval of the difference**, not just the
p-value. A tight interval around a small difference is itself a result.


In [ ]:
!pip install -q transformers sentencepiece accelerate scipy deep-significance

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os, json, itertools
sys.path.insert(0, "/content/drive/MyDrive/google_colab/kusa/v2_heldout")
from config import *
from models import *
import utils_split as u

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from scipy.stats import binomtest, ttest_rel
from tqdm import tqdm

# ---- Execution guard ---------------------------------------------------
FORCE = False
RESULTS_CSV = os.path.join(TEST_EVAL, "test_results.csv")
if os.path.exists(RESULTS_CSV) and not FORCE:
    print("test_results.csv already exists:", RESULTS_CSV)
    raise SystemExit("Aborting. The test set is evaluated exactly once. "
                     "Set FORCE = True to deliberately repeat.")

# ---- Integrity check -------------------------------------------------
with open(MANIFEST, encoding="utf-8") as f:
    MAN = json.load(f)

actual = u.sha256(TEST_LOCKED)
print("Test SHA-256 expected:", MAN["test_locked_sha256"][:32], "...")
print("Test SHA-256 found:", actual[:32], "...")
assert actual == MAN["test_locked_sha256"], (
    "The test set differs from the manifest. Either the split was redrawn "
    "or the file was modified - a valid evaluation is not possible.")
print("OK - test set unchanged since build_split_v2.")

# The FOLD models are loaded, not a single final model.
for v in VARIANTS:
    for k in range(1, N_FOLDS + 1):
        fp = fold_model(v, k)
        assert os.path.exists(fp), f"fold model missing: {fp}"
print(f"OK - all {len(VARIANTS) * N_FOLDS} fold models present.")

In [ ]:
test_df = pd.read_csv(TEST_LOCKED, encoding="utf-8")
test_df = test_df.loc[:, ~test_df.columns.str.contains("^Unnamed")]
test_df = test_df.dropna(subset=["surface", "lemma"]).reset_index(drop=True)

assert len(test_df) == MAN["n_test"], "row count differs from the manifest"
print(f"Test set: {len(test_df)} rows")
print("Labels    :", test_df["label"].value_counts().sort_index().to_dict())
print("Categories:", test_df["category"].value_counts().to_dict())

BERT_MODEL_NAME = "xlm-roberta-large"
MAX_LEN = 128
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

PARAMS = {}
for v in VARIANTS:
    with open(best_params_path(v), encoding="utf-8") as f:
        PARAMS[v] = json.load(f)
    print(f"{v:20s} batch={PARAMS[v]['batch_size']} epochs={PARAMS[v]['epochs']}")

In [ ]:
# Dataset classes and architecture - identical to the CV notebooks.
def score_test(variant, state_path):
    """Load one saved model (fold or final) and score the whole test set once."""
    P = PARAMS[variant]
    surface_only = variant in ("baseline_v2", "single_view_complex_v2")

    if variant == "baseline_v2":
        model = AutoModelForSequenceClassification.from_pretrained(
            BERT_MODEL_NAME, num_labels=3, classifier_dropout=P["dropout"])
    elif variant == "single_view_complex_v2":
        model = SingleViewCNNBiLSTM(
            bert_model_name=BERT_MODEL_NAME, num_classes=3, dropout=P["dropout"])
    else:
        model = DualViewCNNBiLSTMAttention(
            bert_model_name=BERT_MODEL_NAME, num_classes=3,
            dropout=P["dropout"], gated=(variant == "dual_view_gated_v2"))

    if surface_only:
        loader = DataLoader(SurfaceOnlyDataset(test_df, tokenizer),
                            batch_size=P["batch_size"], shuffle=False)
    else:
        loader = DataLoader(DualViewSurfaceLemmaDataset(test_df, tokenizer),
                            batch_size=P["batch_size"], shuffle=False,
                            collate_fn=dual_collator)

    state = torch.load(state_path, map_location="cpu")
    missing, unexpected = model.load_state_dict(state, strict=False)
    assert not missing and not unexpected, (
        f"weights do not match: missing={missing}, unexpected={unexpected}")
    model.to(device).eval()

    preds, probs = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=os.path.basename(state_path), leave=False):
            if variant == "baseline_v2":
                out = model(input_ids=batch["input_ids"].to(device),
                            attention_mask=batch["attention_mask"].to(device))
                logits = out.logits
            elif variant == "single_view_complex_v2":
                logits = model({"input_ids": batch["input_ids"].to(device),
                                "attention_mask": batch["attention_mask"].to(device)})
            else:
                s = {k: v.to(device) for k, v in batch["surface"].items()}
                l = {k: v.to(device) for k, v in batch["lemma"].items()}
                o = model(s, l)
                logits = o[0] if isinstance(o, tuple) else o
            probs.extend(F.softmax(logits, dim=1).cpu().numpy().tolist())
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())

    del model
    torch.cuda.empty_cache()
    return np.array(preds), np.array(probs)

## Scoring

From here on the test set is touched. Each of the five fold models per
architecture scores it exactly once (pure inference, no selection).


In [ ]:
y_true = test_df["label"].values

FOLD_PREDS, FOLD_PROBS, FOLD_F1 = {}, {}, {}

for v in VARIANTS:
    print(f"\n{'='*60}\n{v}\n{'='*60}")
    preds_list, probs_list, f1_list = [], [], []
    for k in range(1, N_FOLDS + 1):
        p, pr = score_test(v, fold_model(v, k))
        preds_list.append(p)
        probs_list.append(pr)
        f1_list.append(f1_score(y_true, p, average="macro"))
        print(f"  fold {k}: macro-F1 {f1_list[-1]:.4f}")

    FOLD_PREDS[v] = np.stack(preds_list)        # (n_folds, n_test)
    FOLD_PROBS[v] = np.stack(probs_list)        # (n_folds, n_test, 3)
    FOLD_F1[v]    = np.array(f1_list)
    print(f"  -> mean {FOLD_F1[v].mean():.4f} +/- {FOLD_F1[v].std(ddof=1):.4f}")

    # Per-fold test predictions, so the pairing over items stays reproducible.
    out = {"row_id": test_df["row_id"].values,
           "category": test_df["category"].values, "label": y_true}
    for k in range(N_FOLDS):
        out[f"pred_fold{k+1}"] = FOLD_PREDS[v][k]
    for j in range(3):
        out[f"meanprob_{j}"] = FOLD_PROBS[v][:, :, j].mean(axis=0)
    pd.DataFrame(out).to_csv(
        os.path.join(TEST_EVAL, f"test_fold_predictions_{v}.csv"),
        index=False, encoding="utf-8")

print("\nPer-fold test predictions saved.")

In [ ]:
# Soft vote (mean of probabilities) and majority vote per architecture.
def ensemble_pred(v):
    return FOLD_PROBS[v].mean(axis=0).argmax(axis=1)

def majority_pred(v):
    P = FOLD_PREDS[v]                       # (n_folds, n_test)
    out = np.zeros(P.shape[1], dtype=int)
    for i in range(P.shape[1]):
        vals, cnts = np.unique(P[:, i], return_counts=True)
        out[i] = vals[np.argmax(cnts)]      # ties -> smallest label, deterministic
    return out

ENS = {v: ensemble_pred(v) for v in VARIANTS}
MAJ = {v: majority_pred(v) for v in VARIANTS}
print("Soft vote and majority vote computed.")

## Level 1 - descriptive

Macro-F1 of the five fold models per architecture (mean +/- std), plus the
ensemble values (soft vote and majority vote). The std over the five folds is
the seed spread that every small effect must survive.


In [ ]:
desc_rows = []
for v in VARIANTS:
    f = FOLD_F1[v]
    desc_rows.append({
        "variant": v,
        "fold_mean_f1": f.mean(),
        "fold_std_f1":  f.std(ddof=1),
        "fold_min_f1":  f.min(),
        "fold_max_f1":  f.max(),
        "ensemble_f1":  f1_score(y_true, ENS[v], average="macro"),
        "majority_f1":  f1_score(y_true, MAJ[v], average="macro"),
        "ensemble_acc": accuracy_score(y_true, ENS[v]),
        "n_test": len(y_true),
    })
desc_df = pd.DataFrame(desc_rows).set_index("variant")
print("===== Level 1: spread on the test set =====")
print(desc_df.round(4).to_string())
desc_df.round(6).to_csv(os.path.join(TEST_EVAL, "test_descriptive.csv"),
                        encoding="utf-8")

for v in VARIANTS:
    print(f"\n{v}: Folds " + ", ".join(f"{x:.4f}" for x in FOLD_F1[v]))

## Level 2 - paired over seeds

The five fold models of both architectures share the same fold assignment and
the same per-fold seed offset. The per-fold difference is therefore paired. The
paired t-test checks whether the mean fold difference is separable from zero;
the ASO test (deep-significance) is the current NLP standard for comparing two
models over several random runs and assumes no normal distribution.

ASO yields `eps_min` in [0, 1]: small (< 0.5, clearly < 0.2) means the first
architecture almost stochastically dominates the second. With only five runs
ASO is conservative - the t-test is the primary number here, ASO the robust
cross-check.


In [ ]:
try:
    from deepsig import aso
    HAVE_ASO = True
except Exception as e:
    HAVE_ASO = False
    print("deep-significance not available, ASO skipped:", e)

def _aso(x, y):
    # deep-significance signatures vary across versions; degrade gracefully.
    try:
        return round(float(aso(x, y, seed=ASO_SEED, show_progress_bar=False)), 4)
    except TypeError:
        return round(float(aso(x, y)), 4)
    except Exception as ex:
        print("  ASO failed:", ex)
        return float("nan")

seed_rows = []
for a, b in itertools.combinations(VARIANTS, 2):
    fa, fb = FOLD_F1[a], FOLD_F1[b]
    d = fa - fb
    mean_d = float(d.mean())
    se = float(d.std(ddof=1) / np.sqrt(len(d)))
    t = ttest_rel(fa, fb)
    row = {"model_a": a, "model_b": b,
           "mean_fold_diff": round(mean_d, 4),
           "se": round(se, 4),
           "t_stat": round(float(t.statistic), 4),
           "p_paired_t": round(float(t.pvalue), 4),
           "n_folds": len(d)}
    if HAVE_ASO:
        row["aso_a_over_b"] = _aso(fa, fb)
        row["aso_b_over_a"] = _aso(fb, fa)
    seed_rows.append(row)

    print(f"\n{a}  vs  {b}")
    print(f"  mean fold difference : {mean_d:+.4f}  (SE {se:.4f})")
    print(f"  paired t-test p      : {t.pvalue:.4f}")
    if HAVE_ASO:
        print(f"  ASO(a>b) eps_min        : {row['aso_a_over_b']:.4f}")
        print(f"  ASO(b>a) eps_min        : {row['aso_b_over_a']:.4f}")

seed_df = pd.DataFrame(seed_rows)
seed_df.to_csv(os.path.join(TEST_EVAL, "significance_seed_level.csv"),
               index=False, encoding="utf-8")
print("\n" + seed_df.to_string(index=False))

## Level 3 - paired over items

Both architectures score the same test items, so the comparison is paired. Both
tests run on the **same** predictions - the soft-vote ensemble, which is also
the headline number - so that they describe one quantity rather than two. The
bootstrap resamples items with replacement and yields the confidence interval of
the macro-F1 difference; McNemar tests only the discordant pairs. The majority
vote is still reported descriptively, but nothing is tested on it.


In [ ]:
def paired_bootstrap(y, pa, pb, B=BOOTSTRAP_B, seed=BOOTSTRAP_SEED):
    rng = np.random.default_rng(seed)
    n = len(y)
    obs = f1_score(y, pa, average="macro") - f1_score(y, pb, average="macro")
    diffs = np.empty(B)
    for i in range(B):
        idx = rng.integers(0, n, n)
        if len(np.unique(y[idx])) < 3:
            diffs[i] = np.nan
            continue
        diffs[i] = (f1_score(y[idx], pa[idx], average="macro")
                    - f1_score(y[idx], pb[idx], average="macro"))
    diffs = diffs[~np.isnan(diffs)]
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    p = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    return obs, lo, hi, min(p, 1.0), len(diffs)


def mcnemar_exact(y, pa, pb):
    a_ok, b_ok = (pa == y), (pb == y)
    b = int((a_ok & ~b_ok).sum())        # only A correct
    c = int((~a_ok & b_ok).sum())        # only B correct
    if b + c == 0:
        return b, c, 1.0
    return b, c, binomtest(min(b, c), b + c, 0.5, alternative="two-sided").pvalue


item_rows = []
for a, b in itertools.combinations(VARIANTS, 2):
    obs, lo, hi, p_boot, B_eff = paired_bootstrap(y_true, ENS[a], ENS[b])
    nb_, nc_, p_mc = mcnemar_exact(y_true, ENS[a], ENS[b])
    item_rows.append({
        "model_a": a, "model_b": b,
        "macro_f1_diff": round(obs, 4),
        "ci_lo": round(lo, 4), "ci_hi": round(hi, 4),
        "p_bootstrap": round(p_boot, 4),
        "only_a_correct": nb_, "only_b_correct": nc_,
        "discordant": nb_ + nc_,
        "p_mcnemar": round(p_mc, 4),
        "n_bootstrap": B_eff,
    })
    print(f"\n{a}  vs  {b}   (both tests on the soft-vote ensemble)")
    print(f"  macro-F1 difference : {obs:+.4f}  (95% CI {lo:+.4f} to {hi:+.4f})")
    print(f"  Bootstrap p        : {p_boot:.4f}")
    print(f"  discordant pairs  : {nb_ + nc_}  (only A: {nb_}, only B: {nc_})")
    print(f"  McNemar p (exact)  : {p_mc:.4f}")
    print("  -> " + ("CI excludes 0" if (lo > 0 or hi < 0)
                     else "CI contains 0 - not separable from noise"))

item_df = pd.DataFrame(item_rows)
item_df.to_csv(os.path.join(TEST_EVAL, "significance_item_level.csv"),
               index=False, encoding="utf-8")
print("\n" + item_df.to_string(index=False))

## Cross-check: categories

Does the test set confirm the category finding from the dev pool? On the
averaged predictions (soft vote) of the baseline. Small categories cannot be
evaluated reliably here.


In [ ]:
from math import sqrt

def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan, np.nan)
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = (z*sqrt(p*(1-p)/n + z*z/(4*n*n))) / d
    return p, max(0, c-h), min(1, c+h)

VAR_FOR_CAT = "baseline_v2"
err = ENS[VAR_FOR_CAT] != y_true

rows = []
for cat in sorted(test_df["category"].unique()):
    m = (test_df["category"] == cat).values
    n = int(m.sum()); e = int(err[m].sum())
    p, lo, hi = wilson(e, n)
    rows.append({"category": cat, "n": n, "errors": e,
                 "error_rate_%": round(100*p, 2),
                 "ci_lo_%": round(100*lo, 2), "ci_hi_%": round(100*hi, 2),
                 "reliable": n >= 200})

cat_df = pd.DataFrame(rows).sort_values("error_rate_%", ascending=False)
print(f"Fehlerrate je Kategorie auf dem Test-Set ({VAR_FOR_CAT}, Soft-Vote)")
print(f"Overall error rate: {100*err.mean():.2f}%\n")
print(cat_df.to_string(index=False))
cat_df.to_csv(os.path.join(TEST_EVAL, "test_error_rate_by_category.csv"),
              index=False, encoding="utf-8")

## Optional: final full-pool model

Only if `RUN_FINAL = True` was set in the CV notebooks and
`final_models/*_final.pth` exist. A single seed - no spread, not part of the
seed-paired tests, shown here only as an extra row for comparison.


In [ ]:
final_rows = []
for v in VARIANTS:
    fp = final_model(v)
    if os.path.exists(fp):
        p, pr = score_test(v, fp)
        final_rows.append({"variant": v,
                           "final_macro_f1": f1_score(y_true, p, average="macro"),
                           "final_accuracy": accuracy_score(y_true, p)})
if final_rows:
    final_df = pd.DataFrame(final_rows).set_index("variant")
    print("Final full-pool models (optional, single seed):")
    print(final_df.round(4).to_string())
    final_df.round(6).to_csv(os.path.join(TEST_EVAL, "test_final_models.csv"),
                             encoding="utf-8")
else:
    print("No final_models/*_final.pth present - skipped (RUN_FINAL was off).")

## Summary for the paper


In [ ]:
# Place dev CV (from cv_summary.csv) next to the test spread.
summary = desc_df.copy()
summary["cv_mean"] = np.nan
summary["cv_std"]  = np.nan
for v in VARIANTS:
    cvp = os.path.join(cv_dir(v), "cv_summary.csv")
    if os.path.exists(cvp):
        s = pd.read_csv(cvp)["macro_f1"]
        summary.loc[v, "cv_mean"] = s.mean()
        summary.loc[v, "cv_std"]  = s.std(ddof=1)

summary = summary[["n_test", "cv_mean", "cv_std",
                   "fold_mean_f1", "fold_std_f1", "ensemble_f1", "ensemble_acc"]]
summary.columns = ["n_test", "cv_dev_mean", "cv_dev_std",
                   "test_fold_mean", "test_fold_std", "test_ensemble_f1",
                   "test_ensemble_acc"]
summary.round(4).to_csv(RESULTS_CSV, encoding="utf-8")

print("===== Table 2: dev CV vs. test (with spread) =====")
print(summary.round(4).to_string())
print(f"\nsaved:")
for f in ["test_results.csv", "test_descriptive.csv",
          "significance_seed_level.csv", "significance_item_level.csv"]:
    print("   ", os.path.join(TEST_EVAL, f))

print("\nThe test set is now spent. Further evaluations on it would"
      "\nno longer be a held-out evaluation.")